# Watermark U-Net — full self-contained run (train + eval)

Upload this notebook to Kaggle, Settings → Accelerator → **GPU**, then run top to bottom.
Every cell is idempotent: re-running after a dead session skips finished work.
Rules: no `&`/`nohup` (Kaggle forbids it), no Save-Version mid-train, download `models/*.pt` every few hours.

In [ ]:
# ===== CELL 0 — parameters (edit ONLY here) + GPU + disk gate =====
EPOCHS, BATCH, SIZE, LR, PATIENCE = 80, 12, 384, 2e-4, 15
USE_LOGO = False  # True only with 20GB+ free (15GB download)
N_SYNTH = 5000
MIN_FREE_GB = 8
import shutil
import torch
print('cuda:', torch.cuda.is_available())
free_gb = shutil.disk_usage('/kaggle/working').free / 1e9
print(f'free disk: {free_gb:.1f} GB')
assert torch.cuda.is_available(), 'STOP: no GPU — set Accelerator to GPU first'
assert free_gb >= MIN_FREE_GB, f'STOP: need {MIN_FREE_GB}GB free, have {free_gb:.1f}'
print('gate OK — continue')

In [ ]:
# ===== CELL 1 — repo + deps (skips if already cloned) =====
import os
os.chdir('/kaggle/working')
get_ipython().system('test -d hamrah-watermark/.git && (cd hamrah-watermark && git pull) || git clone https://github.com/AliTabibAzar/hamrah-watermark.git hamrah-watermark')
os.chdir('/kaggle/working/hamrah-watermark')
get_ipython().system('pip install -q -r requirements-train.txt albumentations rapidocr-onnxruntime gdown')

In [ ]:
# ===== CELL 2 — previous weights from attached dataset (skip if absent) =====
# Attach first: Add Data -> your watermark-unet dataset. Then run.
import glob, os
os.makedirs('models', exist_ok=True)
cands = glob.glob('/kaggle/input/*/watermark-unet*.pt')
print('found in /kaggle/input:', cands if cands else 'NONE (fresh start — fine)')
get_ipython().system('cp /kaggle/input/*/watermark-unet*.pt models/ 2>/dev/null; ls -la models/*.pt 2>/dev/null || echo "no weights yet"')
get_ipython().system('python -c "import torch, glob; f=sorted(glob.glob(\'models/watermark-unet*.pt\')); sd=torch.load(f[0], map_location=\'cpu\') if f else None; print(\'weights OK,\', len(sd), \'tensors\') if sd else print(\'no weights to verify\')"')

In [ ]:
# ===== CELL 2b — PROBE: measure current weights before training (5 min) =====
# LR decision table (edit LR in CELL 0 accordingly, then run CELL 8):
#   probe IoU > 0.5  -> weights are good, fine-tune only: LR = 1e-4
#   probe IoU 0.3-0.5 -> halfway: LR = 2e-4
#   probe IoU < 0.3   -> weak/random: LR = 3e-4
# Needs data/clwd_test (CELL 4). Skip if no weights yet.
get_ipython().system('test -f models/watermark-unet.pt && test -d data/clwd_test/images && python scripts/eval_bulk.py --data data/clwd_test --mode unet --limit 200 --thresholds 0.5 --weights models/watermark-unet.pt || echo "probe skipped (need weights + data/clwd_test first)"')

In [ ]:
# ===== CELL 3 — PITA (skip if converted). Frees raw afterwards. =====
import os
need = not (os.path.isdir('data/pita/images') and len(os.listdir('data/pita/images')) > 10000)
print('PITA needed:', need)
if need:
    get_ipython().system('python scripts/download_data.py --out data --sets pita')
    get_ipython().system('python scripts/convert_pita.py --src data/pita_raw --dst data/pita')
    get_ipython().system('rm -rf data/pita_raw data/pita/_unzipped')
get_ipython().system('ls data/pita/images | wc -l')

In [ ]:
# ===== CELL 4 — CLWD test (skip if converted). Keeps zip for train later. =====
import os
need = not (os.path.isdir('data/clwd_test/images') and len(os.listdir('data/clwd_test/images')) > 5000)
print('CLWD-test needed:', need)
if need:
    get_ipython().system('test -f clwd.zip || gdown --fuzzy "https://drive.google.com/file/d/17y1gkUhIV6rZJg1gMG-gzVMnH27fm4Ij/view?usp=sharing" -O clwd.zip')
    get_ipython().system("unrar x clwd.zip 'CLWD/test/*' data/clwd_raw/")
    get_ipython().system('python scripts/convert_clwd.py --src data/clwd_raw/CLWD/test --dst data/clwd_test')
    get_ipython().system('rm -rf data/clwd_raw')
get_ipython().system('ls data/clwd_test/images | wc -l')

In [ ]:
# ===== CELL 5 — CLWD train, 60K pairs (skip if converted) =====
import os
need = not (os.path.isdir('data/clwd_train/images') and len(os.listdir('data/clwd_train/images')) > 50000)
print('CLWD-train needed:', need)
if need:
    get_ipython().system('test -f clwd.zip || gdown --fuzzy "https://drive.google.com/file/d/17y1gkUhIV6rZJg1gMG-gzVMnH27fm4Ij/view?usp=sharing" -O clwd.zip')
    get_ipython().system("unrar x clwd.zip 'CLWD/train/Mask/*' data/clwd_train_raw/")
    get_ipython().system("unrar x clwd.zip 'CLWD/train/Watermarked_image/*' data/clwd_train_raw/")
    get_ipython().system('python scripts/convert_clwd.py --src data/clwd_train_raw/CLWD/train --dst data/clwd_train')
    get_ipython().system('rm -rf data/clwd_train_raw')
get_ipython().system('ls data/clwd_train/images | wc -l')

In [ ]:
# ===== CELL 6 — LOGO eval (default OFF, needs ~20GB free) =====
import shutil
free_gb = shutil.disk_usage('/kaggle/working').free / 1e9
print(f'USE_LOGO={USE_LOGO} free={free_gb:.1f}GB')
import os
if USE_LOGO and not (os.path.isdir('data/logo_eval/images') and len(os.listdir('data/logo_eval/images')) > 1000):
    assert free_gb >= 20, 'STOP: LOGO needs 20GB free'
    get_ipython().system('python -c "from huggingface_hub import hf_hub_download; print(hf_hub_download(\'vinthony/watermark-removal-logo\', filename=\'10kmid.zip\', repo_type=\'dataset\', local_dir=\'data/logo_dl\'))"')
    get_ipython().system('python scripts/convert_logo.py --zip data/logo_dl/10kmid.zip --dst data/logo_eval')
    get_ipython().system('rm -rf data/logo_dl')
    get_ipython().system('ls data/logo_eval/images | wc -l')

In [ ]:
# ===== CELL 7 — synthetic top-up (skip if present) =====
import os
need = not (os.path.isdir('data/synth/images') and len(os.listdir('data/synth/images')) >= N_SYNTH)
print('synth needed:', need)
if need:
    get_ipython().system(f'python scripts/gen_synthetic.py --bg data/clwd_train/images --out data/synth --n {N_SYNTH}')

In [ ]:
# ===== CELL 8 — TRAIN (foreground; refresh-safe via checkpoints) =====
# LR comes from CELL 0 — set it from the CELL 2b probe table first!
# STOP rule: val IoU < 0.3 past epoch 15 -> stop cell, send the log.
# Expect 'resumed from ...' in the first lines (else STOP: wrong weights path).
# Download models/watermark-unet.pt every few hours from Output.
get_ipython().system(f'python train_unet.py --data data/clwd_train data/synth --out models --epochs {EPOCHS} --batch {BATCH} --size {SIZE} --lr {LR} --patience {PATIENCE} --scheduler cosine --resume')

In [ ]:
# ===== CELL 9 — eval gate (CLWD must PASS; LOGO only if enabled) =====
# Bars: CLWD IoU>=0.45 + recall>=0.70 | LOGO IoU>=0.40.
import os
roots = 'data/clwd_test' + (' data/logo_eval' if (USE_LOGO and os.path.isdir('data/logo_eval/images')) else '')
print('eval roots:', roots)
get_ipython().system(f'python scripts/eval_bulk.py --data {roots} --mode ensemble --limit 500 --thresholds 0.3 0.5 0.7 --weights models/watermark-unet.pt')

In [ ]:
# ===== CELL 10 — export checklist (DO THIS before deleting the notebook) =====
# [ ] models/watermark-unet.pt downloaded (this is v2!) and added to the Kaggle dataset
# [ ] epoch log copied (best val IoU + epoch number)
# [ ] eval GATE line + table sent for review
get_ipython().system('ls -la models/*.pt')